# Practical 7: Inverted Index and Whoosh Search

**Aim:** Build an inverted index and search text files with Boolean and TF-IDF ranked retrieval.

The notebook reads `.txt` files from `data/`. Install Whoosh if needed: `python -m pip install Whoosh`. Run the notebook with the practical folder as the working directory, then run the cells from top to bottom.


In [ ]:
import re
from collections import defaultdict
from pathlib import Path

from whoosh import scoring
from whoosh.fields import ID, TEXT, Schema
from whoosh.index import create_in
from whoosh.qparser import OrGroup, QueryParser

DATA = Path("data")
INDEX = Path("whoosh_index")
documents = [(file.stem, file.read_text(encoding="utf-8"))
             for file in sorted(DATA.glob("*.txt"))]


## 1. Make the inverted index

A word maps to the IDs of files that contain it.


In [ ]:
inverted_index = defaultdict(list)
for doc_id, text in documents:
    for word in set(re.findall(r"\w+", text.lower())):
        inverted_index[word].append(doc_id)

for word in sorted(inverted_index):
    print(word, "->", inverted_index[word])


## 2. Let Whoosh index the files

The schema gives each document an ID and searchable text. The writer adds all files to the Whoosh index.


In [ ]:
INDEX.mkdir(exist_ok=True)
schema = Schema(doc_id=ID(stored=True), content=TEXT(stored=True))
index = create_in(str(INDEX), schema)
with index.writer() as writer:
    for doc_id, text in documents:
        writer.add_document(doc_id=doc_id, content=text)


## 3. Boolean search

Both words in the query must be present. Edit the query string to try other words.


In [ ]:
with index.searcher() as searcher:
    parser = QueryParser("content", schema=index.schema)
    results = searcher.search(parser.parse("machine AND learning"))
    for result in results:
        print(result["doc_id"], result["content"])


## 4. TF-IDF ranked search

This query matches either word, then ranks the matching documents by TF-IDF score.


In [ ]:
with index.searcher(weighting=scoring.TF_IDF()) as searcher:
    parser = QueryParser("content", schema=index.schema, group=OrGroup)
    results = searcher.search(parser.parse("python data"))
    for result in results:
        print(result["doc_id"], round(result.score, 2), result["content"])


## Viva questions with answers

1. **What is an inverted index?** A structure that maps each word to the documents containing it.
2. **What is a postings list?** The document ID list stored for a word.
3. **Why use an inverted index?** It lets a search engine find matching documents without scanning every document for every query.
4. **What does Boolean `AND` mean?** All query terms joined by `AND` must appear in a matching document.
5. **What is ranked retrieval?** Returning matching documents ordered by a relevance score.
6. **What does TF-IDF stand for?** Term Frequency–Inverse Document Frequency.
7. **Why use TF-IDF?** It gives more weight to terms frequent in a document but uncommon across the collection.
8. **What is a Whoosh schema?** The definition of document fields to store and search.
9. **What does `QueryParser` do?** It converts a query string into a query Whoosh can search.
10. **How do the two searches differ here?** Boolean search requires both terms; ranked search can match either term and orders matches by TF-IDF score.
